# Redis Deep Dive 🔥

## What you will learn in this lecture 🧐🧐

In this lecture, you’ll get hands-on with **Redis** — the de-facto Key–Value store for analytics engineers.
You’ll learn how Redis stores data **in memory**, explore its **core data types**, and understand how to manipulate them through Python for real-world analytics use cases.

By the end of this lecture, you’ll be able to:

* Understand how Redis commands (`SET`, `HSET`, `LPUSH`, etc.) work
* Use Python to store, retrieve, and manipulate different Redis data types
* Know how Redis stores data in memory and what happens when memory fills up
* Understand how persistence works in Redis, and how it differs from saving data in a traditional SQL database
* Build efficient caching, counters, and real-time analytics layers on top of Redis

## Redis Commands — Where They Come From ⚙️

Redis exposes a simple **command-line protocol** — every operation is triggered by a command like `SET`, `HSET`, or `LPUSH`.
You can run these commands:

* Interactively with the **Redis CLI** (`redis-cli`)
* Or programmatically through a **client library**, such as **`redis-py`** in Python

Example via CLI:

```bash
SET user:42 "Leia Organa"
GET user:42
```

Example via Python:

In [ ]:
import redis

HOSTNAME = "redis-10004.crce282.eu-west-3-1.ec2.cloud.redislabs.com"
PORT = 10004
PASSWORD = "xxxxxx" # replace with your actual password

r = redis.Redis(
    host=HOSTNAME,
    port=PORT,
    password=PASSWORD,
    decode_responses=True
)

r.set("user-101", "Leia Organa")
print(r.get("user-101"))  # Leia Organa

Leia Organa


<Note type="note">

Redis doesn’t have directories, folders, or subkeys — everything lives in one flat keyspace.
The colons `:` are purely **a naming convention** 🧩.

### 🧠 Think of it like this:

Redis keys are just strings:

```python
"stats:region:eu"
"stats:region:us"
"stats:user:42"
```

Redis doesn’t treat the `:` as special — it’s just a **character**.
But humans (and tools like `redis-cli --scan`) use colons to create **namespaces**, so you can logically organize data.

### 🗂 Example analogy

| Redis key         | What it *represents*                | Type   |
| ----------------- | ----------------------------------- | ------ |
| `user:42`         | user profile for ID 42              | hash   |
| `stats:region:eu` | metrics for Europe                  | hash   |
| `stats:region:us` | metrics for the U.S.                | hash   |
| `session:42`      | temporary login session for user 42 | string |

They’re all separate keys in the same Redis database — just like having multiple variables with descriptive names.

### 📦 Why people use colons

* To group related keys visually
* To make scanning easier:

  ```bash
  redis-cli keys "stats:*"
  ```
returns all stats-related keys.

</Note>



Every Python method (`r.set`, `r.hset`, `r.lpush`) corresponds directly to a Redis command.
When you call them, Python sends a command over the network to the Redis server — and Redis executes it instantly in memory.

<Note type="tip" title="How Redis executes commands">

Redis runs **single-threaded**, executing one command at a time extremely fast.  
This guarantees that your commands are **atomic** — no two operations ever overlap.

</Note>

## Data Types You’ll Actually Use 🧱

Redis is more than a Key–Value store — it’s a **data structure server**.
Each value can be stored as a specific data structure optimized for a certain kind of operation.

### **Strings** — Simple Values & Counters

Strings are the most common data type.
They store text, numbers, or serialized objects.

In [22]:
#r.set("page:home:views", 0)       # store a value
r.incr("page:home:views")         # increment by 1
print(r.get("page:home:views"))   # returns '1'

# cache a value with expiration
r.setex("kpi:avg_order_value", 300, "58.9")  # expires in 5 minutes

20


True

<Note type="tip" title="When to use">

Use Strings for **simple values**, **counters**, or **cached query results**.

</Note>

### **Hashes** — Structured Records (User Profiles, Metadata)

Hashes store multiple fields and values under a single key, similar to a Python dictionary.


In [49]:
r.hset("user:43", mapping={
    "name": "Han Solo",
    "tier": "gold",
    "last_seen": "2025-10-16T08:00Z"
})

print(r.hgetall("user:43"))

r.hincrby("stats:region:eu", "orders", 200)

{'name': 'Han Solo', 'tier': 'gold', 'last_seen': '2025-10-16T08:00Z'}


686

Use Hashes for **small, structured data** like user profiles, configuration, or region stats.

<Note type="important" title="edit keys">

You can edit an existing key, but **only if you keep the same data type**.
If you try to change the type of value (e.g. from a string 👉 hash), Redis refuses. You'll get a `WRONGTYPE` error.


### ✅ What you *can* do

You can update a key’s content **as long as the type stays the same**.

#### Example 1 – Strings

```python
r.set("user:42", "Leia Organa")
r.set("user:42", "General Organa")  # ✅ Overwrites string value
print(r.get("user:42"))  # b'General Organa'
```

#### Example 2 – Hashes

```python
r.hset("user:42", mapping={"name": "Leia", "tier": "gold"})
r.hset("user:42", "last_seen", "2025-10-16T08:00Z")  # ✅ Adds/updates a field
print(r.hgetall("user:42"))
# {b'name': b'Leia', b'tier': b'gold', b'last_seen': b'2025-10-16T08:00Z'}
```

Redis lets you freely mutate the contents — but only within that same structure.

### ❌ What you *can’t* do

You **cannot change a key’s type** once created, unless you delete it first.

#### Example

```python
r.set("user:42", "Leia Organa")
r.hset("user:42", mapping={"name": "Leia"})  # ❌ WRONGTYPE
```

Redis is type-safe:

> “Each key in Redis belongs to exactly one data type.”

If you want to reuse the same key name for a new type, you must delete it first:

```python
r.delete("user:42")
r.hset("user:42", mapping={"name": "Leia"})  # ✅ now works
```

</Note>

### **Lists** — Ordered Collections (Logs, Queues)

Lists are ordered sequences.
You can push items from the left or right and pop them later.


In [50]:
# Add new checkout events
r.lpush("events:checkout", '{"user": 42, "amount": 79.9}')
r.lpush("events:checkout", '{"user": 43, "amount": 51.3}')

# Get the 2 most recent
print(r.lrange("events:checkout", 0, 1))
# ['{"user": 43, "amount": 51.3}', '{"user": 42, "amount": 79.9}']

# Keep only last 1000 events
r.ltrim("events:checkout", 0, 999)

['{"user": 43, "amount": 51.3}', '{"user": 42, "amount": 79.9}']


True


<Note type="tip" title="When to use">

Use Lists for **recent events**, **job queues**, or **activity feeds**.

</Note>

### **Sets** — Unique Elements (Membership & Deduplication)

Sets hold unique values — perfect for tracking unique visitors or active users.


In [51]:
r.sadd("active_users", 41, 42, 43)
print(r.smembers("active_users"))     # {'41', '42', '43'}
print(r.sismember("active_users", 42)) # True
print(r.scard("active_users"))         # 3

{'41', '43', '42'}
1
3


<Note type="tip" title="When to use">

Use Sets to track **unique IDs**, **active sessions**, or **distinct visitors**.

</Note>

### **Sorted Sets (ZSETs)** — Rankings & Time Windows

Sorted Sets store elements with scores and automatically keep them sorted.
Useful for leaderboards, KPIs, or ranking systems.


In [52]:

r.zadd("leaderboard", {"han": 1200, "luke": 1350, "leia": 1100})

# Get top 2 players
print(r.zrevrange("leaderboard", 0, 1, withscores=True))
# [('luke', 1350.0), ('han', 1200.0)]

r.zincrby("leaderboard", 50, "leia")  # update Leia’s score

[('luke', 1350.0), ('han', 1200.0)]


1150.0

<Note type="tip" title="When to use">

Use ZSETs for **leaderboards**, **ranked KPIs**, or **rolling time-based metrics**.

</Note>

### **Streams** — Lightweight Event Logs

Streams let you record time-ordered events and consume them later, similar to Kafka but simpler.


In [53]:
r.xadd("clicks", {"user": "42", "page": "/home"})
r.xadd("clicks", {"user": "43", "page": "/pricing"})

# Read first 2 events
print(r.xrange("clicks", "-", "+", count=2))
# [('<id1>', {'user': '42', 'page': '/home'}), ('<id2>', {'user': '43', 'page': '/pricing'})]

[('1773658129261-0', {'user': '42', 'page': '/home'}), ('1773658129268-0', {'user': '43', 'page': '/pricing'})]



<Note type="tip" title="When to use">

Use Streams for **event tracking**, **ETL pipelines**, or **real-time logs**.

</Note>

### **What About Blobs?**

In Redis, values can also be **blobs** — arbitrary binary data like images, compressed JSON, or serialized Python objects.


In [54]:
import pickle # For serializing complex data

r = redis.Redis(host=HOSTNAME, port=PORT, password=PASSWORD, decode_responses=False)  # 👈 important change decode_responses to False

data = {"user": 44, "features": [0.1, 0.5, 0.8]} # Let's say this is a ML feature vector
r.set("user:44:features", pickle.dumps(data))

retrieved = pickle.loads(r.get("user:44:features"))
print(retrieved)

{'user': 44, 'features': [0.1, 0.5, 0.8]}


In [55]:
pickle.dumps(data)

b'\x80\x04\x958\x00\x00\x00\x00\x00\x00\x00}\x94(\x8c\x04user\x94K,\x8c\x08features\x94]\x94(G?\xb9\x99\x99\x99\x99\x99\x9aG?\xe0\x00\x00\x00\x00\x00\x00G?\xe9\x99\x99\x99\x99\x99\x9aeu.'


Redis doesn’t interpret blob contents — it just stores and retrieves them very quickly.

## Reference Cheat-Sheet 🧠

* **Strings:** `SET`, `GET`, `INCR`, `DECR`, `SETEX`, `MGET`
* **Hashes:** `HSET`, `HGET`, `HGETALL`, `HINCRBY`
* **Lists:** `LPUSH`, `RPUSH`, `LRANGE`, `LTRIM`
* **Sets:** `SADD`, `SCARD`, `SISMEMBER`, `SMEMBERS`
* **Sorted Sets:** `ZADD`, `ZINCRBY`, `ZREVRANGE`, `ZREMRANGEBYSCORE`
* **Streams:** `XADD`, `XRANGE`, `XREAD`
* **Keys/Server:** `DEL`, `EXPIRE`, `TTL`, `SCAN`, `INFO`

## Understanding “In-Memory” Storage 🧩

Redis stores all data in **RAM**, not on disk — that’s why it’s extremely fast (microseconds latency).

When you call `r.set("key", "value")`, the data is kept directly in memory, allowing instant access without disk reads.

However, RAM is limited. Redis cannot exceed the physical memory available on your server.

### Example — 16 GB Server

If each entry (key + metadata + value) averages **200 bytes**,
then a 16 GB Redis server can store roughly **80 million entries**.

That’s enough for dashboards, caches, and metrics — but not for large historical datasets.

### When Memory Fills Up

Redis uses an **eviction policy** to decide what to delete when memory is full:

| Policy         | Behavior                                      |
| -------------- | --------------------------------------------- |
| `volatile-lru` | Remove least recently used keys **with TTLs** |
| `allkeys-lru`  | Remove least recently used keys globally      |
| `noeviction`   | Stop accepting new writes (default)           |

You can configure this:

```bash
CONFIG SET maxmemory-policy allkeys-lru
```

### Strategies for Limited Memory

* Use **TTLs** for caches or temporary data.
* **Shard** data across multiple Redis instances.
* **Move cold data** to slower storage like PostgreSQL or S3.
* Monitor memory usage with `INFO memory`.

<Note type="important" title="Redis is your cache, not your warehouse">
Redis is your **speed layer** — for hot data that needs instant access.  
Cold or historical data belongs in a **persistent storage system**.
</Note>

## TTL, Expiration & Self-Cleaning Keys

TTL (Time To Live) allows Redis to delete data automatically after a certain time.


In [56]:
r.setex("cache:report:eu", 600, "serialized-result")
print(r.ttl("cache:report:eu"))  # remaining lifetime in seconds

600



<Note type="tip" title="When to use TTLs">

Use TTLs to control cache freshness — short (1–5 min) for dynamic KPIs, longer (15–60 min) for heavy reports.

</Note>

## Pipelines, Transactions & Lua Scripting 🧠

Redis supports batching and atomic execution to ensure operations are efficient and consistent.

### Pipelines — Send Multiple Commands at Once

Normally, every Redis command requires a separate network round-trip.
Pipelines **batch multiple commands together**, reducing latency.


In [14]:
pipe = r.pipeline()
for region in ["eu", "us", "apac"]:
    pipe.incr(f"counter:region:{region}")
results = pipe.execute()
print(results)

[1, 1, 1]



✅ All commands are sent in one go, which is much faster.

<Note type="tip" title="When to use">

Use Pipelines for **bulk updates** or **batch writes**.

</Note>


### Transactions — All-or-Nothing Updates

Transactions ensure a group of commands either all succeed or all fail, preserving consistency.


In [15]:
with r.pipeline(transaction=True) as pipe:
    pipe.multi()
    pipe.incr("budget:eu")
    pipe.decr("budget:us")
    pipe.execute()


If one command fails, none of the others apply.

<Note type="tip" title="When to use">

Use transactions when you need **consistency** across multiple keys —  
for example, transferring values between two accounts.

</Note>

### Lua Scripting — Execute Logic on the Server

Lua scripts allow custom logic to run directly inside Redis, atomically and efficiently.


In [16]:
script = r.register_script("""
local key = KEYS[1]
local delta = tonumber(ARGV[1])
local value = tonumber(redis.call('GET', key) or '0')
value = value + delta
redis.call('SET', key, value)
return value
""")

print(script(keys=["kpi:revenue"], args=[5]))

5


All steps run directly on the server — preventing race conditions.

<Note type="tip" title="When to use Lua">

Use Lua for **conditional updates**, **rolling counters**, or **multi-step operations** that must happen atomically.

</Note>

## Persistence & Durability 💾

Redis is primarily **in-memory**, but it can save data to disk using persistence mechanisms.
This ensures that if the server restarts, your data can be restored.

| Mode                       | Description                                                                      |
| -------------------------- | -------------------------------------------------------------------------------- |
| **RDB (Snapshot)**         | Redis periodically saves a full copy of its dataset to disk (e.g., every 5 min). |
| **AOF (Append Only File)** | Redis logs every write operation to disk, allowing precise recovery.             |
| **Hybrid**                 | Combines RDB snapshots for speed + AOF logs for reliability.                     |

### How is this different from a SQL Database?

A SQL database **always writes directly to disk** — every insert or update is durable by default.
Redis, on the other hand, **writes to memory first** and **decides later** if and when to persist.

This is what makes Redis so fast — but also what makes it less reliable if persistence is not enabled.

| Scenario                              | SQL DB | Redis                      |
| ------------------------------------- | ------ | -------------------------- |
| Every write saved to disk immediately | ✅      | ❌ (optional, slower)       |
| Keeps all history                     | ✅      | ❌ (unless logged via AOF)  |
| Survives full reboot automatically    | ✅      | ✅ (if persistence enabled) |
| Tuned for long-term storage           | ✅      | ❌ (in-memory focus)        |

### When to Persist in Redis

* Persist data that cannot be recomputed (financial data, critical counters).
* Use **AOF** if durability matters — it logs every command.
* Use **RDB** for lighter persistence or cache recovery.

### When to Transfer Data to a Real Database

* When data grows beyond your memory capacity.
* When you need **queries, joins, or historical analytics**.
* When data must be **archived, auditable, or compliant** (GDPR, logs, etc.).

Redis is your “frontline memory,” SQL is your “source of truth.”
A common architecture is to **write to both**:
Redis for speed, SQL for durability.

<Note type="important" title="Practical rule of thumb">

Persist only what’s critical.  
Everything else can live in Redis temporarily and be rebuilt if lost.

</Note>

## Monitoring & Debugging

| Command                          | Description                     |
| -------------------------------- | ------------------------------- |
| `INFO memory`                    | View memory usage               |
| `INFO keyspace`                  | View key statistics             |
| `SCAN 0 MATCH pattern COUNT 100` | Iterate safely over keys        |
| `MONITOR`                        | View live commands (debug only) |

You can also use **RedisInsight**, a free GUI tool to visualize keys, memory, and TTLs.

## Resources 📚📚

* [Redis Official Documentation](https://redis.io/docs/latest/)
* [Redis Data Types Explained](https://redis.io/docs/latest/develop/data-types/)
* [Redis University — Data Structures & Streams](https://university.redis.com/)
* [Designing Data-Intensive Applications – Martin Kleppmann, Ch. 3–4](https://dataintensive.net/)
* [RedisInsight GUI](https://redis.io/insight/)
* [Caching Strategies Explained (Redis Blog)](https://redis.io/learn/develop/java/cache/cache-strategies)


In [20]:
import pprint
pprint.pprint(r.info("memory"))
pprint.pprint(r.info("keyspace"))

{'maxmemory_policy': 'volatile-lru',
 'mem_allocator': 'jemalloc-5.3.0',
 'mem_fragmentation_ratio': 1,
 'used_memory': 2684136,
 'used_memory_human': '2.55M',
 'used_memory_lua': 34816,
 'used_memory_peak': 2702504,
 'used_memory_peak_human': '2.57M',
 'used_memory_rss': 2684136}
{'db0': {'avg_ttl': 524400, 'expires': 1, 'keys': 17}}
